In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# --- LOAD REAL DATA ---
raw_df = pd.read_csv('5-min-breakdown-All-Final.csv')

# --- TIME ENCODING ---
# Convert Time strings (e.g. "5'", "HT", "FT") to numeric minutes
def parse_time(t):
    if t == 'HT':
        return 45
    elif t == 'FT':
        return 90
    else:
        return int(t.replace("'", ""))

raw_df['time'] = raw_df['Time'].apply(parse_time)

# --- DERIVED FEATURES ---
raw_df['score diff'] = raw_df['Home_Score'] - raw_df['Away_Score']
raw_df['home subs remaining'] = 5 - raw_df['Home_Total_Sub_Count']
raw_df['away subs remaining'] = 5 - raw_df['Away_Total_Sub_Count']
raw_df['red diff'] = raw_df['Home_Red_Count'] - raw_df['Away_Red_Count']

# --- MATCH TYPE ENCODING ---
le_game_type = LabelEncoder()
raw_df['match type'] = le_game_type.fit_transform(raw_df['Game_Type'])

# --- TARGET ENCODING ---
# Encode match outcome: 0 = Away Win, 1 = Draw, 2 = Home Win
outcome_map = {'AWAY_WIN': 0, 'DRAW': 1, 'HOME_WIN': 2}
raw_df['match outcome'] = raw_df['Match_Outcome'].map(outcome_map)

# --- SELECT & RENAME FEATURES TO MATCH MODEL SCHEMA ---
df = raw_df[[
    'time',                        # time
    'Away_Score',                  # away goals
    'Home_Score',                  # home goals
    'score diff',                  # score diff
    'Home_Offensive_Sub_Count',    # home attk sub
    'Home_Defensive_Sub_Count',    # home def sub
    'Away_Offensive_Sub_Count',    # away attk sub
    'Away_Defensive_Sub_Count',    # away def sub
    'home subs remaining',         # home subs remaining
    'away subs remaining',         # away subs remaining
    'Home_Red_Count',              # home red count
    'Home_First_Red_Time',         # home time 1st red
    'Away_Red_Count',              # away red count
    'Away_First_Red_Time',         # away time first red
    'red diff',                    # red diff
    'match type',                  # match type
    'Odds_Home_Win',               # pre-match home win
    'Odds_Draw',                   # pre-match draw
    'Odds_Away_Win',               # pre-match away win
    'Home_Yellow_Count',           # home yellow count
    'Away_Yellow_Count',           # away yellow count
    'match outcome'                # target
]].copy()

df.columns = [
    'time', 'away goals', 'home goals', 'score diff',
    'home attk sub', 'home def sub', 'away attk sub', 'away def sub',
    'home subs remaining', 'away subs remaining',
    'home red count', 'home time 1st red', 'away red count', 'away time first red',
    'red diff', 'match type',
    'pre-match home win', 'pre-match draw', 'pre-match away win',
    'home yellow count', 'away yellow count',
    'match outcome'
]

# --- FILL MISSING RED TIMES WITH 0 (no red card = 0) ---
df['home time 1st red'] = df['home time 1st red'].fillna(0)
df['away time first red'] = df['away time first red'].fillna(0)

print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")
print(f"Match outcome distribution:\n{df['match outcome'].value_counts().rename({0:'Away Win', 1:'Draw', 2:'Home Win'})}")
display(df.head())

Loaded 105,819 rows x 22 columns
Match outcome distribution:
match outcome
Home Win    45591
Away Win    33159
Draw        27069
Name: count, dtype: int64


,time,away goals,home goals,score diff,home attk sub,home def sub,away attk sub,away def sub,home subs remaining,away subs remaining,...,away red count,away time first red,red diff,match type,pre-match home win,pre-match draw,pre-match away win,home yellow count,away yellow count,match outcome
0,0,0,0,0,0,0,0,0,5,5,...,0,0.0,0,0,2.63,3.16,2.87,0,0,0
1,5,0,0,0,0,0,0,0,5,5,...,0,0.0,0,0,2.63,3.16,2.87,0,0,0
2,10,0,0,0,0,0,0,0,5,5,...,0,0.0,0,0,2.63,3.16,2.87,0,0,0
3,15,0,0,0,0,0,0,0,5,5,...,0,0.0,0,0,2.63,3.16,2.87,0,0,0
4,20,1,0,-1,0,0,0,0,5,5,...,0,0.0,0,0,2.63,3.16,2.87,0,0,0


In [3]:
# 1. Separate features and target
target_col = 'match outcome'

# 2. Encode categorical variables if necessary (e.g., match type)
# Assuming 'match type' might be categorical in your real data
if df['match type'].dtype == 'object':
    le = LabelEncoder()
    df['match type'] = le.fit_transform(df['match type'])

# 3. Custom Grouped Split (90/10 per time/score combination)
train_list = []
test_list = []

# Group by 'time' and 'score diff'
grouped = df.groupby(['time', 'score diff'])

for name, group in grouped:
    # If a group is too small to split, put it entirely in train (or handle otherwise)
    if len(group) < 2:
        train_list.append(group)
        continue

    # 90/10 split for this specific combination
    train_group, test_group = train_test_split(group, test_size=0.10, random_state=42)
    train_list.append(train_group)
    test_list.append(test_group)

# Concatenate back into full train and test sets
train_df = pd.concat(train_list).reset_index(drop=True)
test_df = pd.concat(test_list).reset_index(drop=True)

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

# 4. Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train_scaled.shape}")
print(f"Testing set size: {X_test_scaled.shape}")

Training set size: (95153, 21)
Testing set size: (10666, 21)


In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

# Determine number of unique classes in target
num_classes = len(np.unique(y_train))

# Build the Neural Network
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu'),

    # Output layer: using softmax for multi-class classification (Home Win, Draw, Away Win)
    Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # Use sparse if y is label-encoded integers
    metrics=['accuracy']
)

model.summary()

# Train the model
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test_scaled, y_test),
    verbose=1
)

# Evaluate
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         2,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,019 (54.76 KB)

 Trainable params: 13,635 (53.26 KB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/30
2974/2974 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.3133 - loss: nan - val_accuracy: 0.3149 - val_loss: nan
Epoch 2/30
2974/2974 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.3132 - loss: nan - val_accuracy: 0.3149 - val_loss: nan
Epoch 3/30
2974/2974 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.3132 - loss: nan - val_accuracy: 0.3149 - val_loss: nan
Epoch 4/30
2974/2974 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - accuracy: 0.3132 - loss: nan - val_accuracy: 0.3149 - val_loss: nan
Epoch 5/30
2974/2974 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.3132 - loss: nan - val_accuracy: 0.3149 - val_loss: nan
Epoch 6/30
2974/2974 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.3132 - loss: nan - val_accuracy: 0.3149 - val_loss: nan
Epoch 7/30
2974/2974 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.3132 - loss: nan - val_accuracy: 0.3149 - val_loss: nan
Epoch 8/30
2974/2974 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.3132 - loss: nan - val_accuracy: 0.3149 - val_loss: nan
Ep

In [6]:
from sklearn.metrics import accuracy_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 1. Get predictions for the test set
y_pred_prob = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_prob, axis=1)

# 2. Add predictions back to a copy of the test dataframe
results_df = test_df.copy()
results_df['predicted outcome'] = y_pred
results_df['actual outcome'] = y_test

# 3. Clamp score diff to -4 to +4
results_df['clamped_score_diff'] = results_df['score diff'].clip(lower=-4, upper=4)

# 4. Calculate metrics for each time & score diff permutation
metrics_list = []

grouped_results = results_df.groupby(['time', 'clamped_score_diff'])

for (time_bin, score_diff), group in grouped_results:
    y_true_g = group['actual outcome']
    y_pred_g = group['predicted outcome']

    # Calculate metrics (using macro average for multi-class recall/f1)
    acc = accuracy_score(y_true_g, y_pred_g)
    rec = recall_score(y_true_g, y_pred_g, average='macro', zero_division=0)
    f1 = f1_score(y_true_g, y_pred_g, average='macro', zero_division=0)

    metrics_list.append({
        'time': time_bin,
        'clamped_score_diff': score_diff,
        'accuracy': acc,
        'recall': rec,
        'f1_score': f1,
        'count': len(group)
    })

metrics_df = pd.DataFrame(metrics_list)

# 5. Generate a pivot table for accuracy
accuracy_pivot = metrics_df.pivot(index='clamped_score_diff', columns='time', values='accuracy')

# Sort index descending so +4 is at top, -4 at bottom
accuracy_pivot = accuracy_pivot.sort_index(ascending=False)

# 6. Style the table
# Define a custom colormap: red < 0.35 < green
import matplotlib.colors as mcolors

# We want 0 to 0.35 to map from Red to White, and 0.35 to 1.0 to map from White to Green
norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=0.35, vmax=1.0)
cmap = mcolors.LinearSegmentedColormap.from_list(
    'custom_rg',
    [(0.0, 'red'), (0.35, 'white'), (1.0, 'green')]
)

styled_table = accuracy_pivot.style.background_gradient(cmap=cmap, axis=None, vmin=0, vmax=1.0) \
    .format("{:.2f}", na_rep="-") \
    .set_caption("Accuracy by Time and Score Differential (-4 to +4)")

display(styled_table)

# Print overall metrics to ensure we have them
print("\n--- Overall Metrics for Context ---")
print(f"Overall Accuracy: {accuracy_score(results_df['actual outcome'], results_df['predicted outcome']):.4f}")
print(f"Overall Recall (Macro): {recall_score(results_df['actual outcome'], results_df['predicted outcome'], average='macro'):.4f}")
print(f"Overall F1 (Macro): {f1_score(results_df['actual outcome'], results_df['predicted outcome'], average='macro'):.4f}")


334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


time,0,5,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90
clamped_score_diff,,,,,,,,,,,,,,,,,,,
4,-,-,-,-,-,-,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,-,-,-,0.00,0.00,0.00,0.00,0.00,0.17,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,-,-,0.33,0.00,0.00,0.00,0.05,0.04,0.00,0.03,0.00,0.00,0.00,0.00,0.07,0.02,0.02,0.02,0.00
1,-,0.19,0.11,0.12,0.11,0.10,0.15,0.11,0.11,0.14,0.07,0.07,0.07,0.07,0.06,0.04,0.05,0.05,0.00
0,0.34,0.31,0.34,0.32,0.30,0.33,0.30,0.30,0.26,0.30,0.19,0.31,0.28,0.17,0.23,0.16,0.15,0.17,0.00
-1,-,0.61,0.42,0.51,0.55,0.43,0.51,0.56,0.62,0.59,0.56,0.60,0.66,0.66,0.68,0.72,0.72,0.77,1.00
-2,-,1.00,0.50,1.00,0.75,0.77,0.59,0.80,0.80,0.84,0.72,0.79,0.82,0.82,0.95,0.90,0.88,0.98,1.00
-3,-,-,-,1.00,1.00,1.00,1.00,1.00,1.00,0.92,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
-4,-,-,-,-,-,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00



--- Overall Metrics for Context ---
Overall Accuracy: 0.3149
Overall Recall (Macro): 0.3333
Overall F1 (Macro): 0.1597
